In [1]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture # To draw circles
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm, SqrtStretch, ZScaleInterval
import os # For creating output directory

# --- Configuration ---
input_dir = Path("data/stretched_output/good")
output_dir = Path("data/detected_stars_visualizations") # Directory to save output images
output_dir.mkdir(parents=True, exist_ok=True) # Create output directory if it doesn't exist

# DAOStarFinder parameters (ADJUST THESE!)
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Brightness threshold criterion
brightness_threshold_percent = 90.0 # Keep stars >= 90% of the max peak brightness

# Visualization settings
marker_color = 'red'
marker_radius = 15 # Radius of the circle marker in pixels

# --- Find image files ---
# Using .jpg previews as requested. Change to .xisf for better accuracy.
image_files = list(input_dir.glob("*_stretched_preview.jpg"))
# To use XISF instead (recommended):
# import xisf
# image_files = list(input_dir.glob("*_stretched.xisf"))

if not image_files:
    print(f"!!! ERROR: No JPEG preview files found in {input_dir}. !!!")
    print("Please check the directory path and ensure the preview files exist.")
else:
    print(f"Found {len(image_files)} images to process in {input_dir}")
    print(f"Saving visualizations to: {output_dir}")

    # --- Process each image ---
    processed_count = 0
    errors_count = 0
    for image_path in image_files:
        print(f"\n--- Processing: {image_path.name} ---")
        try:
            # 1. Load the image (grayscale)
            # --- Loading JPEG ---
            gray_image_uint8 = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if gray_image_uint8 is None:
                raise IOError(f"Could not load image file: {image_path}")
            display_image = cv2.cvtColor(gray_image_uint8, cv2.COLOR_GRAY2RGB) # For color plotting later
            gray_image = gray_image_uint8.astype(float) # Convert to float for calculations
            # --- ---

            # --- (Alternative) Loading XISF ---
            # if image_path.suffix == '.xisf':
            #     image_data_raw = xisf.XISF.read(image_path)
            #     # Convert to grayscale if RGB XISF
            #     if image_data_raw.ndim == 3 and image_data_raw.shape[2] == 3:
            #         gray_image_uint16 = cv2.cvtColor(image_data_raw, cv2.COLOR_RGB2GRAY)
            #     elif image_data_raw.ndim == 2:
            #         gray_image_uint16 = image_data_raw
            #     else:
            #          raise ValueError(f"Unsupported XISF shape: {image_data_raw.shape}")
            #     # Normalize 16-bit to 8-bit for display, keep float for detection
            #     display_image_8bit = cv2.normalize(gray_image_uint16, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
            #     display_image = cv2.cvtColor(display_image_8bit, cv2.COLOR_GRAY2RGB)
            #     gray_image = gray_image_uint16.astype(float) # Use 16-bit float for detection
            # --- ---

            # 2. Estimate background and noise
            mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
            if std == 0: std = np.finfo(gray_image.dtype).eps # Avoid division by zero
            print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")

            # 3. Run DAOStarFinder
            data_subtracted = gray_image - median
            daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
            sources = daofind(data_subtracted)

            if sources is None:
                print("  No stars found with current settings.")
                continue # Skip to the next image

            print(f"  Detected {len(sources)} potential stars initially.")

            # 4. Filter for bright stars
            max_peak = sources['peak'].max()
            brightness_cutoff = max_peak * (brightness_threshold_percent / 100.0)
            bright_sources_mask = sources['peak'] >= brightness_cutoff
            bright_sources = sources[bright_sources_mask]

            if len(bright_sources) == 0:
                print(f"  No stars found above brightness threshold ({brightness_cutoff:.2f}).")
                continue # Skip to the next image

            print(f"  Found {len(bright_sources)} stars >= {brightness_threshold_percent}% of max peak ({max_peak:.2f}).")

            # 5. Create visualization
            positions = np.transpose((bright_sources['xcentroid'], bright_sources['ycentroid']))
            apertures = CircularAperture(positions, r=marker_radius)

            # Create plot
            plt.figure(figsize=(12, 8)) # Adjust figure size as needed
            # Display image using a robust stretch like ZScale
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(gray_image) # Use original gray image for limits
            norm = simple_norm(gray_image, stretch='asinh', min_cut=vmin, max_cut=vmax)
            plt.imshow(display_image, origin='lower', cmap='gray', norm=norm, interpolation='nearest') # Display color or gray
            apertures.plot(color=marker_color, lw=1.5, alpha=0.8) # Draw circles

            plt.title(f"{image_path.name}\nDetected {len(bright_sources)} bright stars (Peak >= {brightness_cutoff:.1f})", fontsize=10)
            plt.colorbar(label='Pixel Value (Approx)') # Add a colorbar
            plt.xlabel("X pixel")
            plt.ylabel("Y pixel")

            # Save the figure
            output_filename = output_dir / f"{image_path.stem}_detected.jpg"
            plt.savefig(output_filename, bbox_inches='tight', dpi=150)
            plt.close() # Close the figure to free memory
            print(f"  Saved visualization to: {output_filename.name}")
            processed_count += 1

        except Exception as e:
            print(f"!!! ERROR processing {image_path.name}: {e} !!!")
            errors_count += 1
            # Optional: Add detailed traceback printing here if needed
            # import traceback
            # traceback.print_exc()

    print(f"\n--- Finished processing ---")
    print(f"Successfully processed and saved visualizations for {processed_count} images.")
    if errors_count > 0:
        print(f"Encountered errors on {errors_count} images.")

Found 279 images to process in data/stretched_output/good
Saving visualizations to: data/detected_stars_visualizations

--- Processing: 2024-10-14_04-47-19__-0.00_120.00s_0015_stretched_preview.jpg ---
  Background Median: 19.00, Noise (Std Dev): 6.77
  Detected 2433 potential stars initially.
  Found 358 stars >= 90.0% of max peak (236.00).


  Saved visualization to: 2024-10-14_04-47-19__-0.00_120.00s_0015_stretched_preview_detected.jpg

--- Processing: 2024-10-20_06-01-35_Great Orion Nebula_-0.00_30.00s_0309_stretched_preview.jpg ---
  Background Median: 14.00, Noise (Std Dev): 4.83
  Detected 1281 potential stars initially.
  Found 176 stars >= 90.0% of max peak (241.00).


  Saved visualization to: 2024-10-20_06-01-35_Great Orion Nebula_-0.00_30.00s_0309_stretched_preview_detected.jpg

--- Processing: 2024-10-14_05-01-36__-0.00_30.00s_0114_stretched_preview.jpg ---
  Background Median: 5.00, Noise (Std Dev): 1.74
  Detected 2814 potential stars initially.
  Found 146 stars >= 90.0% of max peak (250.00).


  Saved visualization to: 2024-10-14_05-01-36__-0.00_30.00s_0114_stretched_preview_detected.jpg

--- Processing: 2024-10-21_04-10-28_Great Orion Nebula_-0.00_30.00s_0373_stretched_preview.jpg ---
  Background Median: 10.00, Noise (Std Dev): 3.54
  Detected 1843 potential stars initially.
  Found 149 stars >= 90.0% of max peak (245.00).


  Saved visualization to: 2024-10-21_04-10-28_Great Orion Nebula_-0.00_30.00s_0373_stretched_preview_detected.jpg

--- Processing: 2025-06-10_00-24-19_North America Nebula_-0.00_300.00s_0015_stretched_preview.jpg ---
  Background Median: 20.00, Noise (Std Dev): 5.87
  Detected 8136 potential stars initially.
  Found 240 stars >= 90.0% of max peak (235.00).


...

  Saved visualization to: 2024-10-14_04-38-24__0.00_120.00s_0011_stretched_preview_detected.jpg

--- Processing: 2024-10-14_04-49-20__0.00_120.00s_0016_stretched_preview.jpg ---
  Background Median: 20.00, Noise (Std Dev): 7.02
  Detected 2356 potential stars initially.
  Found 357 stars >= 90.0% of max peak (235.00).


  Saved visualization to: 2024-10-14_04-49-20__0.00_120.00s_0016_stretched_preview_detected.jpg

--- Processing: 2024-10-21_04-01-06_Great Orion Nebula_-0.00_30.00s_0357_stretched_preview.jpg ---
  Background Median: 10.00, Noise (Std Dev): 3.60
  Detected 1779 potential stars initially.
  Found 126 stars >= 90.0% of max peak (245.00).


  Saved visualization to: 2024-10-21_04-01-06_Great Orion Nebula_-0.00_30.00s_0357_stretched_preview_detected.jpg

--- Processing: 2025-06-09_23-25-37_North America Nebula_-0.00_300.00s_0004_stretched_preview.jpg ---
  Background Median: 25.00, Noise (Std Dev): 7.35
  Detected 5757 potential stars initially.
  Found 241 stars >= 90.0% of max peak (230.00).


  Saved visualization to: 2025-06-09_23-25-37_North America Nebula_-0.00_300.00s_0004_stretched_preview_detected.jpg

--- Processing: 2024-10-20_03-52-49_Great Orion Nebula_0.00_30.00s_0238_stretched_preview.jpg ---
  Background Median: 15.00, Noise (Std Dev): 4.95


KeyboardInterrupt: 

In [2]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
import os
import math

# --- Configuration ---
input_dir = Path("data/stretched_output/good")
# --- NEW: Save 90% cutouts to a separate directory ---
output_dir = Path("data/star_cutouts_bright_90pct_pages")
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0
brightness_threshold_percent = 90.0 # <-- Your 90% setting

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Find image files ---
image_files = list(input_dir.glob("*_stretched_preview.jpg"))

if not image_files:
    print(f"!!! ERROR: No JPEG preview files found in {input_dir}. !!!")
else:
    print(f"Found {len(image_files)} images to process.")
    print(f"Saving 'Top {brightness_threshold_percent}%' cutout pages to: {output_dir}")

    # --- Process each image file ---
    total_processed_files = 0
    total_errors = 0
    for image_path in image_files:
        print(f"\n--- Processing Image: {image_path.name} ---")
        try:
            # 1. Load the JPEG image as grayscale
            gray_image_uint8 = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if gray_image_uint8 is None:
                raise IOError(f"Could not load image file: {image_path}")
            gray_image = gray_image_uint8.astype(float) # Use float for calculations
            print(f"  Image loaded: shape={gray_image.shape}")

            # 2. Estimate background and noise
            mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
            print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
            if std == 0: std = np.finfo(gray_image.dtype).eps

            # 3. Run DAOStarFinder to get ALL stars
            data_subtracted = gray_image - median
            daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
            sources = daofind(data_subtracted)

            if sources is None:
                print("  No stars found with current settings. Skipping this image.")
                continue

            print(f"  Detected {len(sources)} potential stars initially.")

            # --- 4. NEW: Filter for 90% brightness ---
            max_peak = sources['peak'].max()
            brightness_cutoff = max_peak * (brightness_threshold_percent / 100.0)
            bright_sources_mask = sources['peak'] >= brightness_cutoff
            bright_sources = sources[bright_sources_mask] # This is our new list
            # --- End of filter ---

            total_bright_stars = len(bright_sources)
            if total_bright_stars == 0:
                print(f"  No stars found above {brightness_threshold_percent}% brightness threshold ({brightness_cutoff:.2f}). Skipping this image.")
                continue

            print(f"  --- Found {total_bright_stars} stars >= {brightness_threshold_percent}% peak. ---")

            # 5. Sort by brightness (descending)
            bright_sources.sort('peak')
            bright_sources.reverse()

            # 6. Paginate and create cutout images FOR THIS FILE
            num_pages = int(math.ceil(total_bright_stars / stars_per_page))
            print(f"  Generating {num_pages} grayscale cutout page(s) for these bright stars...")

            for page_num in range(num_pages):
                start_idx = page_num * stars_per_page
                end_idx = min(total_bright_stars, (page_num + 1) * stars_per_page)
                page_sources = bright_sources[start_idx:end_idx] # <-- Use bright_sources
                num_on_page = len(page_sources)

                # Create the plot grid
                fig, axes = plt.subplots(grid_dim, grid_dim,
                                         figsize=(1.5 * grid_dim, 1.5 * grid_dim),
                                         squeeze=False)
                axes = axes.ravel() # Flatten

                for i, source in enumerate(page_sources):
                    x, y = source['xcentroid'], source['ycentroid']
                    peak = source['peak']

                    # Define cutout boundaries
                    x_min = max(0, int(x - cutout_size // 2))
                    x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                    y_min = max(0, int(y - cutout_size // 2))
                    y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                    cutout = gray_image[y_min:y_max, x_min:x_max]

                    if cutout.size == 0:
                        axes[i].set_visible(False); continue

                    # Display cutout using grayscale colormap
                    ax = axes[i]
                    norm = simple_norm(cutout, stretch='linear', percent=99.0)
                    ax.imshow(cutout, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
                    ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5)
                    ax.set_xticks([]); ax.set_yticks([])

                # Hide unused subplots
                for j in range(num_on_page, len(axes)):
                    axes[j].set_visible(False)

                plt.suptitle(f"Top {brightness_threshold_percent}% Cutouts: Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_path.name})", fontsize=10)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                # Save the figure with a unique name
                output_filename = output_dir / f"{image_path.stem}_bright_90pct_cutouts_page_{page_num + 1:03d}.png"
                plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                plt.close(fig) # Close the figure to free memory

            print(f"  Saved {num_pages} page(s) for this image.")
            total_processed_files += 1

        except Exception as e:
            print(f"!!! ERROR processing {image_path.name}: {e} !!!")
            total_errors += 1
            # import traceback # Uncomment for detailed error traces
            # traceback.print_exc()

    print(f"\n--- Batch Cutout Generation Finished ---")
    print(f"Successfully processed {total_processed_files} images.")
    if total_errors > 0:
        print(f"Encountered errors on {total_errors} images.")
    print(f"All 'Top 90%' cutout pages are saved in: {output_dir}")

Found 279 images to process.
Saving 'Top 90.0%' cutout pages to: data/star_cutouts_bright_90pct_pages

--- Processing Image: 2024-10-14_04-47-19__-0.00_120.00s_0015_stretched_preview.jpg ---
  Image loaded: shape=(6388, 9576)
  Background Median: 19.00, Noise (Std Dev): 6.77
  Detected 2433 potential stars initially.
  --- Found 358 stars >= 90.0% peak. ---
  Generating 4 grayscale cutout page(s) for these bright stars...
  Saved 4 page(s) for this image.

--- Processing Image: 2024-10-20_06-01-35_Great Orion Nebula_-0.00_30.00s_0309_stretched_preview.jpg ---
  Image loaded: shape=(6388, 9576)
  Background Median: 14.00, Noise (Std Dev): 4.83
  Detected 1281 potential stars initially.
  --- Found 176 stars >= 90.0% peak. ---
  Generating 2 grayscale cutout page(s) for these bright stars...
  Saved 2 page(s) for this image.

--- Processing Image: 2024-10-14_05-01-36__-0.00_30.00s_0114_stretched_preview.jpg ---
  Image loaded: shape=(6388, 9576)
  Background Median: 5.00, Noise (Std Dev)

In [ ]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm # Still useful for contrast
import os
import math

# --- Configuration ---
image_file_path = Path("data/stretched_output/good/2024-10-14_04-36-22__-0.00_120.00s_0010_stretched_preview.jpg")
# Save grayscale cutouts to a new directory
output_dir = Path("data/star_cutouts_grayscale_pages") # New output directory
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Star Detection & Cutout Generation ---
print(f"Processing JPEG: {image_file_path}")
print("--- Generating GRAYSCALE cutouts ---")

if not image_file_path.exists():
    print(f"!!! ERROR: File not found: {image_file_path} !!!")
else:
    try:
        # 1. Load the JPEG image as grayscale
        print("Loading grayscale JPEG image...")
        gray_image_uint8 = cv2.imread(str(image_file_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_file_path}")
        print(f"Image loaded: shape={gray_image_uint8.shape}, dtype={gray_image_uint8.dtype}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
        else:
            total_stars = len(sources)
            print(f"\n--- Found {total_stars} total stars ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # 5. Paginate and create cutout images
            num_pages = int(math.ceil(total_stars / stars_per_page))
            print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

            for page_num in range(num_pages):
                start_idx = page_num * stars_per_page
                end_idx = min(total_stars, (page_num + 1) * stars_per_page)
                page_sources = sources[start_idx:end_idx]
                num_on_page = len(page_sources)

                # Create the plot grid
                fig, axes = plt.subplots(grid_dim, grid_dim,
                                         figsize=(1.5 * grid_dim, 1.5 * grid_dim), # Smaller figure size for grayscale
                                         squeeze=False)
                axes = axes.ravel() # Flatten for easy iteration

                for i, source in enumerate(page_sources):
                    x, y = source['xcentroid'], source['ycentroid']
                    peak = source['peak']

                    # Define cutout boundaries
                    x_min = max(0, int(x - cutout_size // 2))
                    x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                    y_min = max(0, int(y - cutout_size // 2))
                    y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                    cutout = gray_image[y_min:y_max, x_min:x_max]

                    if cutout.size == 0:
                        axes[i].set_visible(False)
                        continue

                    # Display cutout using grayscale colormap
                    ax = axes[i]
                    # Apply simple normalization for contrast, but map to gray
                    norm = simple_norm(cutout, stretch='linear', percent=99.0) # Linear stretch might be better for ML data
                    ax.imshow(cutout, norm=norm, origin='lower', cmap='gray', interpolation='nearest') # *** cmap='gray' ***
                    ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                    ax.set_xticks([]) # Hide ticks
                    ax.set_yticks([])

                # Hide unused subplots on the last page
                for j in range(num_on_page, len(axes)):
                    axes[j].set_visible(False)

                plt.suptitle(f"Grayscale Cutouts: Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                # Save the figure
                output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                plt.close(fig) # Close the figure
                print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except FileNotFoundError:
        print(f"!!! ERROR: File not found at {image_file_path} !!!")
    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name}: {e} !!!")
        import traceback
        traceback.print_exc()

print(f"\n--- Grayscale cutout generation finished. Check the '{output_dir}' directory. ---")

In [3]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
import os
import math
import random # <-- Import random library

# --- Configuration ---
# --- NEW: Set base directory to 'component_output' ---
base_input_dir = Path("data/component_output")
# Define the categories you want to sample from
categories = ['focus', 'good', 'light', 'satellite', 'tracking', 'wind']

# --- NEW: Save these category cutouts to a new directory ---
output_dir = Path("data/category_samples_cutouts_bright_90pct")
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0
brightness_threshold_percent = 90.0 # <-- Your 90% setting

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- NEW: Find one random image from each category ---
image_files = []
print("Finding one random preview JPEG from each category...")
for category in categories:
    category_dir = base_input_dir / category
    if not category_dir.is_dir():
        print(f"  Warning: Category directory not found, skipping: {category_dir}")
        continue
    
    # Find all JPEG previews in this category (recursively)
    # Assumes previews are named like '..._preview.jpg'
    # Use rglob to search all subdirectories
    files_in_category = list(category_dir.rglob("*_preview.jpg"))
    
    if files_in_category:
        # Select one random file
        random_file = random.choice(files_in_category)
        image_files.append(random_file)
        print(f"  Selected for '{category}': {random_file.name}")
    else:
        print(f"  Warning: No '*_preview.jpg' files found in '{category}'")
# --- End of new file finding logic ---


if not image_files:
    print(f"\n!!! ERROR: No JPEG preview files were found in any category directories. !!!")
else:
    print(f"\nFound {len(image_files)} total images to process (one per category).")
    print(f"Saving 'Top {brightness_threshold_percent}%' cutout pages to: {output_dir}")

    # --- Process each image file ---
    total_processed_files = 0
    total_errors = 0
    for image_path in image_files:
        print(f"\n--- Processing Image: {image_path.name} (from '{image_path.parent.name}') ---")
        try:
            # 1. Load the JPEG image as grayscale
            # This is correct, as component previews are grayscale
            gray_image_uint8 = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if gray_image_uint8 is None:
                raise IOError(f"Could not load image file: {image_path}")
            gray_image = gray_image_uint8.astype(float) # Use float for calculations
            print(f"  Image loaded: shape={gray_image.shape}")

            # 2. Estimate background and noise
            mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
            print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
            if std == 0: std = np.finfo(gray_image.dtype).eps

            # 3. Run DAOStarFinder to get ALL stars
            data_subtracted = gray_image - median
            daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
            sources = daofind(data_subtracted)

            if sources is None:
                print("  No stars found with current settings. Skipping this image.")
                continue

            print(f"  Detected {len(sources)} potential stars initially.")

            # --- 4. Filter for 90% brightness ---
            max_peak = sources['peak'].max()
            brightness_cutoff = max_peak * (brightness_threshold_percent / 100.0)
            bright_sources_mask = sources['peak'] >= brightness_cutoff
            bright_sources = sources[bright_sources_mask]
            # --- End of filter ---

            total_bright_stars = len(bright_sources)
            if total_bright_stars == 0:
                print(f"  No stars found above {brightness_threshold_percent}% brightness threshold ({brightness_cutoff:.2f}). Skipping this image.")
                continue

            print(f"  --- Found {total_bright_stars} stars >= {brightness_threshold_percent}% peak. ---")

            # 5. Sort by brightness (descending)
            bright_sources.sort('peak')
            bright_sources.reverse()

            # 6. Paginate and create cutout images FOR THIS FILE
            num_pages = int(math.ceil(total_bright_stars / stars_per_page))
            print(f"  Generating {num_pages} grayscale cutout page(s) for these bright stars...")

            for page_num in range(num_pages):
                start_idx = page_num * stars_per_page
                end_idx = min(total_bright_stars, (page_num + 1) * stars_per_page)
                page_sources = bright_sources[start_idx:end_idx]
                num_on_page = len(page_sources)

                fig, axes = plt.subplots(grid_dim, grid_dim,
                                         figsize=(1.5 * grid_dim, 1.5 * grid_dim),
                                         squeeze=False)
                axes = axes.ravel()

                for i, source in enumerate(page_sources):
                    x, y = source['xcentroid'], source['ycentroid']
                    peak = source['peak']

                    x_min = max(0, int(x - cutout_size // 2))
                    x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                    y_min = max(0, int(y - cutout_size // 2))
                    y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                    cutout = gray_image[y_min:y_max, x_min:x_max]

                    if cutout.size == 0:
                        axes[i].set_visible(False); continue

                    ax = axes[i]
                    norm = simple_norm(cutout, stretch='linear', percent=99.0)
                    ax.imshow(cutout, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
                    ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5)
                    ax.set_xticks([]); ax.set_yticks([])

                for j in range(num_on_page, len(axes)):
                    axes[j].set_visible(False)

                plt.suptitle(f"Top {brightness_threshold_percent}% Cutouts: Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_path.name})", fontsize=10)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                # Save the figure with a unique name
                output_filename = output_dir / f"{image_path.stem}_bright_90pct_cutouts_page_{page_num + 1:03d}.png"
                plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                plt.close(fig)

            print(f"  Saved {num_pages} page(s) for this image.")
            total_processed_files += 1

        except Exception as e:
            print(f"!!! ERROR processing {image_path.name}: {e} !!!")
            total_errors += 1
            # import traceback # Uncomment for detailed error traces
            # traceback.print_exc()

    print(f"\n--- Batch Cutout Generation Finished ---")
    print(f"Successfully processed {total_processed_files} images.")
    if total_errors > 0:
        print(f"Encountered errors on {total_errors} images.")
    print(f"All 'Top 90%' cutout pages are saved in: {output_dir}")

Finding one random preview JPEG from each category...
  Selected for 'focus': 2024-10-14_03-54-19__-0.00_30.00s_0087_stretched_R_preview.jpg
  Selected for 'good': 2025-06-11_01-06-50_North America Nebula_-0.00_300.00s_0020_stretched_G_preview.jpg
  Selected for 'light': 2025-06-10_05-23-10_North America Nebula_-0.00_300.00s_0063_stretched_B_preview.jpg
  Selected for 'satellite': 2024-10-14_04-10-02__-0.00_30.00s_0109_stretched_G_preview.jpg
  Selected for 'tracking': 2025-06-11_02-57-59_North America Nebula_-0.00_300.00s_0038_stretched_R_preview.jpg
  Selected for 'wind': 2024-10-21_05-30-42_Great Orion Nebula_-0.00_30.00s_0381_stretched_G_preview.jpg

Found 6 total images to process (one per category).
Saving 'Top 90.0%' cutout pages to: data/category_samples_cutouts_bright_90pct

--- Processing Image: 2024-10-14_03-54-19__-0.00_30.00s_0087_stretched_R_preview.jpg (from 'focus') ---
  Image loaded: shape=(6388, 9576)
  Background Median: 5.00, Noise (Std Dev): 1.71
  Detected 2414 p

  No stars found with current settings. Skipping this image.

--- Processing Image: 2024-10-14_04-10-02__-0.00_30.00s_0109_stretched_G_preview.jpg (from 'satellite') ---
  Image loaded: shape=(6388, 9576)
  Background Median: 5.00, Noise (Std Dev): 1.79
  Detected 2640 potential stars initially.
  --- Found 158 stars >= 90.0% peak. ---
  Generating 2 grayscale cutout page(s) for these bright stars...
  Saved 2 page(s) for this image.

--- Processing Image: 2025-06-11_02-57-59_North America Nebula_-0.00_300.00s_0038_stretched_R_preview.jpg (from 'tracking') ---
  Image loaded: shape=(6388, 9576)
  Background Median: 35.00, Noise (Std Dev): 10.17
  Detected 5165 potential stars initially.
  --- Found 463 stars >= 90.0% peak. ---
  Generating 5 grayscale cutout page(s) for these bright stars...
  Saved 5 page(s) for this image.

--- Processing Image: 2024-10-21_05-30-42_Great Orion Nebula_-0.00_30.00s_0381_stretched_G_preview.jpg (from 'wind') ---
  Image loaded: shape=(6388, 9576)
  Back

In [5]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm # Still useful for contrast
import os
import math

# --- Configuration ---
# Save grayscale cutouts to a new base directory
# We will create sub-folders for each category inside this
base_output_dir = Path("data/star_cutouts_grayscale_pages_by_category")

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Dictionary of images to process (Category: Filepath) ---
image_list = {
    "good": "data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg",
    "focus": "data/component_output/focus/2024-10-14_03-14-05__-0.00_300.00s_0026_stretched_R_preview.jpg",
    "light": "data/component_output/light/2025-06-10_05-08-04_North America Nebula_-0.00_300.00s_0060_stretched_R_preview.jpg",
    "satellite": "data/component_output/satellite/2024-10-14_04-55-09__-0.00_120.00s_0018_stretched_B_preview.jpg",
    "tracking": "data/component_output/tracking/2025-06-10_02-46-21_North America Nebula_0.00_300.00s_0040_stretched_G_preview.jpg",
    "wind": "data/component_output/wind/2024-10-21_05-57-16_Great Orion Nebula_-0.00_30.00s_0401_stretched_R_preview.jpg"
}

# --- Main Processing Loop ---
print(f"--- Starting GRAYSCALE cutout generation for {len(image_list)} categories ---")
print(f"Base output directory: {base_output_dir}")

for category, image_path_str in image_list.items():
    
    image_file_path = Path(image_path_str)
    
    # Create a new, category-specific output directory
    output_dir = base_output_dir / category
    output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

    # --- Star Detection & Cutout Generation for this file ---
    print("\n" + "="*80)
    print(f"Processing Category: '{category}'")
    print(f"File: {image_file_path}")
    print(f"Outputting to: {output_dir}")
    print("="*80)

    if not image_file_path.exists():
        print(f"!!! ERROR: File not found: {image_file_path} !!!")
        print(f"--- Skipping category '{category}' ---")
        continue # Skip to the next image in the loop
    
    try:
        # 1. Load the JPEG image as grayscale
        print("Loading grayscale JPEG image...")
        gray_image_uint8 = cv2.imread(str(image_file_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_file_path}")
        print(f"Image loaded: shape={gray_image_uint8.shape}, dtype={gray_image_uint8.dtype}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
        else:
            total_stars = len(sources)
            print(f"\n--- Found {total_stars} total stars ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # 5. Paginate and create cutout images
            num_pages = int(math.ceil(total_stars / stars_per_page))
            print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

            for page_num in range(num_pages):
                start_idx = page_num * stars_per_page
                end_idx = min(total_stars, (page_num + 1) * stars_per_page)
                page_sources = sources[start_idx:end_idx]
                num_on_page = len(page_sources)

                # Create the plot grid
                fig, axes = plt.subplots(grid_dim, grid_dim,
                                         figsize=(1.5 * grid_dim, 1.5 * grid_dim), # Smaller figure size for grayscale
                                         squeeze=False)
                axes = axes.ravel() # Flatten for easy iteration

                for i, source in enumerate(page_sources):
                    x, y = source['xcentroid'], source['ycentroid']
                    peak = source['peak']

                    # Define cutout boundaries
                    x_min = max(0, int(x - cutout_size // 2))
                    x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                    y_min = max(0, int(y - cutout_size // 2))
                    y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                    cutout = gray_image[y_min:y_max, x_min:x_max]

                    if cutout.size == 0:
                        axes[i].set_visible(False)
                        continue

                    # Display cutout using grayscale colormap
                    ax = axes[i]
                    # Apply simple normalization for contrast, but map to gray
                    norm = simple_norm(cutout, stretch='linear', percent=99.0) # Linear stretch might be better for ML data
                    ax.imshow(cutout, norm=norm, origin='lower', cmap='gray', interpolation='nearest') # *** cmap='gray' ***
                    ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                    ax.set_xticks([]) # Hide ticks
                    ax.set_yticks([])

                # Hide unused subplots on the last page
                for j in range(num_on_page, len(axes)):
                    axes[j].set_visible(False)

                plt.suptitle(f"Grayscale Cutouts ({category}): Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                # Save the figure
                output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                plt.close(fig) # Close the figure
                print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except FileNotFoundError:
        # This catch is redundant due to the check at the start, but good to have
        print(f"!!! ERROR: File not found at {image_file_path} !!!")
    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name} (Category: {category}): {e} !!!")
        import traceback
        traceback.print_exc()

    print(f"--- Finished processing category '{category}' ---")

print("\n" + "="*80)
print(f"--- ALL cutout generation finished. Check the sub-folders inside '{base_output_dir}'. ---")
print("="*80)

--- Starting GRAYSCALE cutout generation for 6 categories ---
Base output directory: data/star_cutouts_grayscale_pages_by_category

Processing Category: 'good'
File: data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg
Outputting to: data/star_cutouts_grayscale_pages_by_category/good
Loading grayscale JPEG image...
Image loaded: shape=(6388, 9576), dtype=uint8
Estimating background...
  Background Median: 7.00, Noise (Std Dev): 2.38
Running DAOStarFinder (FWHM=4.0, Threshold=5.0*std)...

--- Found 1937 total stars ---
Generating 20 grayscale cutout image pages (100 stars per page)...
  Saved grayscale page 1/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_001.png
  Saved grayscale page 2/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_002.png
  Saved grayscale page 3/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_003.png
  Saved 

In [3]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
import os
import math
import random # <-- Import random library

# --- Configuration ---
# Set base directory to 'component_output'
base_input_dir = Path("data/component_output")
# Define the categories you want to sample from (based on your screenshot)
categories = ['focus', 'good', 'light', 'satellite', 'tracking', 'wind']

# NEW: Save this single comparison plot to a new directory
output_dir = Path("data/visual_comparison")
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 512       # Size of the square cutout box in pixels (50x50)
samples_per_category = 5 # How many random stars to show for each category

# --- Setup the Plot Grid ---
num_categories = len(categories)
fig, axes = plt.subplots(num_categories, samples_per_category,
                         figsize=(2.5 * samples_per_category, 2.5 * num_categories), # 5 cols, 6 rows
                         squeeze=False)

print("--- Generating Category Comparison Plot ---")

# --- Process each category (each row in the plot) ---
for row_idx, category in enumerate(categories):
    print(f"\nProcessing category: {category}")
    
    # Set the Y-axis label for the row
    axes[row_idx, 0].set_ylabel(category.upper(), fontsize=12, fontweight='bold', labelpad=15)
    
    # Find a random file
    category_dir = base_input_dir / category
    if not category_dir.is_dir():
        print(f"  Warning: Category directory not found, skipping: {category_dir}")
        # Hide all subplots for this row
        for col_idx in range(samples_per_category):
            axes[row_idx, col_idx].set_visible(False)
        continue
        
    # Find all JPEG previews in this category (recursively)
    files_in_category = list(category_dir.rglob("*_preview.jpg"))
    
    if not files_in_category:
        print(f"  Warning: No '*_preview.jpg' files found in '{category}'. Skipping.")
        # Hide all subplots for this row
        for col_idx in range(samples_per_category):
            axes[row_idx, col_idx].set_visible(False)
        continue
        
    # Select one random file
    image_path = random.choice(files_in_category)
    print(f"  Using random file: {image_path.name}")

    try:
        # 1. Load the JPEG image as grayscale
        gray_image_uint8 = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_path}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder to get ALL stars
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None or len(sources) == 0:
            print("  No stars found in this file. Skipping row.")
            for col_idx in range(samples_per_category):
                axes[row_idx, col_idx].set_visible(False)
            continue

        # --- 4. NEW: Select N random star samples ---
        num_found = len(sources)
        if num_found <= samples_per_category:
            # Take all found stars if fewer than requested
            selected_sources = sources
        else:
            # Randomly sample if more
            indices = random.sample(range(num_found), samples_per_category)
            selected_sources = sources[indices]
        
        print(f"  Extracting {len(selected_sources)} random star cutouts...")

        # --- 5. Plot cutouts in the row's columns ---
        for col_idx, source in enumerate(selected_sources):
            if col_idx >= samples_per_category: break # Safety break
            
            ax = axes[row_idx, col_idx]
            x, y = source['xcentroid'], source['ycentroid']

            # Define cutout boundaries
            x_min = max(0, int(x - cutout_size // 2))
            x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
            y_min = max(0, int(y - cutout_size // 2))
            y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))
            cutout = gray_image[y_min:y_max, x_min:x_max]

            if cutout.size == 0:
                ax.set_visible(False); continue
            
            # Plot
            norm = simple_norm(cutout, stretch='linear', percent=99.0)
            ax.imshow(cutout, norm=norm, origin='lower', cmap='gray', interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            if row_idx == 0: # Add column title to the very top row
                ax.set_title(f"Sample {col_idx + 1}", fontsize=10)

        # Hide any unused subplots in this row (if fewer than 5 stars were found)
        for col_idx in range(len(selected_sources), samples_per_category):
            axes[row_idx, col_idx].set_visible(False)

    except Exception as e:
        print(f"  !!! ERROR processing {image_path.name}: {e} !!!")

# --- Finalize and save the plot ---
plt.suptitle("Side-by-Side Comparison of Star Cutouts by Category", fontsize=16)
plt.tight_layout(rect=[0.03, 0.03, 1, 0.95]) # Adjust layout to make room for labels

output_filename = output_dir / "category_comparison_grid.png"
plt.savefig(output_filename, bbox_inches='tight', dpi=150)
plt.close(fig)
print(f"\n--- Comparison plot saved to: {output_filename} ---")

--- Generating Category Comparison Plot ---

Processing category: focus
  Using random file: 2024-10-14_03-51-28__-0.00_30.00s_0082_stretched_B_preview.jpg
  Extracting 5 random star cutouts...

Processing category: good
  Using random file: 2024-10-20_04-39-39_Great Orion Nebula_-0.00_30.00s_0269_stretched_B_preview.jpg
  Extracting 5 random star cutouts...

Processing category: light
  Using random file: 2025-06-10_05-18-08_North America Nebula_-0.00_300.00s_0062_stretched_R_preview.jpg


  No stars found in this file. Skipping row.

Processing category: satellite
  Using random file: 2024-10-14_05-11-15__-0.00_30.00s_0124_stretched_G_preview.jpg
  Extracting 5 random star cutouts...

Processing category: tracking
  Using random file: 2025-06-10_04-24-23_North America Nebula_-0.00_300.00s_0057_stretched_B_preview.jpg
  Extracting 5 random star cutouts...

Processing category: wind
  Using random file: 2025-06-10_03-15-55_North America Nebula_-0.00_300.00s_0045_stretched_R_preview.jpg
  Extracting 5 random star cutouts...

--- Comparison plot saved to: data/visual_comparison/category_comparison_grid.png ---


In [ ]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os
import math

# --- Configuration ---

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!  UPDATE THIS PATH to the relative path of your JPG or PNG   !!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
image_file_path = Path("data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg")
# (Using the previous 'good' image as an example)

# Define a category and output dir for this single run
category = "good_filtered_80_percent"
base_output_dir = Path("data/star_cutouts_grayscale_filtered")
output_dir = base_output_dir / category
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Main Processing Block ---
print(f"--- Starting GRAYSCALE cutout generation (Top 80% Brightness) ---")
print(f"Processing Category: '{category}'")
print(f"File: {image_file_path}")
print(f"Outputting to: {output_dir}")
print("="*80)

if not image_file_path.exists():
    print(f"!!! ERROR: File not found: {image_file_path} !!!")
    print(f"--- Please update the 'image_file_path' variable in the script. ---")
else:
    try:
        # 1. Load the JPEG image as grayscale
        print("Loading grayscale JPEG image...")
        gray_image_uint8 = cv2.imread(str(image_file_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_file_path}")
        print(f"Image loaded: shape={gray_image_uint8.shape}, dtype={gray_image_uint8.dtype}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
        else:
            total_stars_found = len(sources)
            print(f"\n--- Found {total_stars_found} total stars (before filtering) ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # --- 5. NEW: Filter by 80% brightness ---
            if total_stars_found > 0:
                max_peak = sources['peak'][0]
                brightness_threshold = max_peak * 0.80
                
                print(f"  Brightest star peak: {max_peak:.2f}")
                print(f"  80% brightness threshold: {brightness_threshold:.2f}")
                
                # Filter the sources table
                sources = sources[sources['peak'] >= brightness_threshold]
                total_stars_filtered = len(sources)
                
                print(f"--- Keeping {total_stars_filtered} stars above 80% brightness ---")
            else:
                print("  No stars found, skipping filtering.")
            # --- End of filtering step ---

            if len(sources) == 0:
                print("\n!!! No stars remaining after 80% brightness filter. !!!")
            else:
                # 6. Paginate and create cutout images (for filtered list)
                num_pages = int(math.ceil(len(sources) / stars_per_page))
                print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

                for page_num in range(num_pages):
                    start_idx = page_num * stars_per_page
                    end_idx = min(len(sources), (page_num + 1) * stars_per_page)
                    page_sources = sources[start_idx:end_idx]
                    num_on_page = len(page_sources)

                    # Create the plot grid
                    fig, axes = plt.subplots(grid_dim, grid_dim,
                                             figsize=(1.5 * grid_dim, 1.5 * grid_dim),
                                             squeeze=False)
                    axes = axes.ravel() # Flatten for easy iteration

                    for i, source in enumerate(page_sources):
                        x, y = source['xcentroid'], source['ycentroid']
                        peak = source['peak']

                        # Define cutout boundaries
                        x_min = max(0, int(x - cutout_size // 2))
                        x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                        y_min = max(0, int(y - cutout_size // 2))
                        y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                        cutout = gray_image[y_min:y_max, x_min:x_max]

                        if cutout.size == 0:
                            axes[i].set_visible(False)
                            continue

                        # Display cutout using grayscale colormap (NO STRETCH)
                        ax = axes[i]
                        ax.imshow(cutout, origin='lower', cmap='gray', interpolation='nearest', vmin=0, vmax=255) 
                        ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                        ax.set_xticks([]) # Hide ticks
                        ax.set_yticks([])

                    # Hide unused subplots on the last page
                    for j in range(num_on_page, len(axes)):
                        axes[j].set_visible(False)

                    plt.suptitle(f"Grayscale Cutouts ({category}): Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                    # Save the figure
                    output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                    plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                    plt.close(fig) # Close the figure
                    print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name} (Category: {category}): {e} !!!")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)
print(f"--- ALL cutout generation finished. Check the folder '{output_dir}'. ---")
print("="*80)

In [6]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
# NOTE: simple_norm removed as requested
import os
import math

# --- Configuration ---
# Save grayscale cutouts to a new base directory
# We will create sub-folders for each category inside this
base_output_dir = Path("data/star_cutouts_grayscale_pages_by_category")

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Dictionary of images to process (Category: Filepath) ---
image_list = {
    "good": "data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg",
    "focus": "data/component_output/focus/2024-10-14_03-14-05__-0.00_300.00s_0026_stretched_R_preview.jpg",
    "light": "data/component_output/light/2025-06-10_05-08-04_North America Nebula_-0.00_300.00s_0060_stretched_R_preview.jpg",
    "satellite": "data/component_output/satellite/2024-10-14_04-55-09__-0.00_120.00s_0018_stretched_B_preview.jpg",
    "tracking": "data/component_output/tracking/2025-06-10_02-46-21_North America Nebula_0.00_300.00s_0040_stretched_G_preview.jpg",
    "wind": "data/component_output/wind/2024-10-21_05-57-16_Great Orion Nebula_-0.00_30.00s_0401_stretched_R_preview.jpg"
}

# --- Main Processing Loop ---
print(f"--- Starting GRAYSCALE cutout generation for {len(image_list)} categories (NO STRETCH) ---")
print(f"Base output directory: {base_output_dir}")

for category, image_path_str in image_list.items():
    
    image_file_path = Path(image_path_str)
    
    # Create a new, category-specific output directory
    output_dir = base_output_dir / category
    output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

    # --- Star Detection & Cutout Generation for this file ---
    print("\n" + "="*80)
    print(f"Processing Category: '{category}'")
    print(f"File: {image_file_path}")
    print(f"Outputting to: {output_dir}")
    print("="*80)

    if not image_file_path.exists():
        print(f"!!! ERROR: File not found: {image_file_path} !!!")
        print(f"--- Skipping category '{category}' ---")
        continue # Skip to the next image in the loop
    
    try:
        # 1. Load the JPEG image as grayscale
        print("Loading grayscale JPEG image...")
        gray_image_uint8 = cv2.imread(str(image_file_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_file_path}")
        print(f"Image loaded: shape={gray_image_uint8.shape}, dtype={gray_image_uint8.dtype}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
        else:
            total_stars = len(sources)
            print(f"\n--- Found {total_stars} total stars ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # 5. Paginate and create cutout images
            num_pages = int(math.ceil(total_stars / stars_per_page))
            print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

            for page_num in range(num_pages):
                start_idx = page_num * stars_per_page
                end_idx = min(total_stars, (page_num + 1) * stars_per_page)
                page_sources = sources[start_idx:end_idx]
                num_on_page = len(page_sources)

                # Create the plot grid
                fig, axes = plt.subplots(grid_dim, grid_dim,
                                         figsize=(1.5 * grid_dim, 1.5 * grid_dim), # Smaller figure size for grayscale
                                         squeeze=False)
                axes = axes.ravel() # Flatten for easy iteration

                for i, source in enumerate(page_sources):
                    x, y = source['xcentroid'], source['ycentroid']
                    peak = source['peak']

                    # Define cutout boundaries
                    x_min = max(0, int(x - cutout_size // 2))
                    x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                    y_min = max(0, int(y - cutout_size // 2))
                    y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                    cutout = gray_image[y_min:y_max, x_min:x_max]

                    if cutout.size == 0:
                        axes[i].set_visible(False)
                        continue

                    # Display cutout using grayscale colormap
                    ax = axes[i]
                    
                    # *** CHANGE: Removed simple_norm stretch ***
                    # Displaying raw pixel values, clamped to 0-255
                    ax.imshow(cutout, origin='lower', cmap='gray', interpolation='nearest', vmin=0, vmax=255) 
                    
                    ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                    ax.set_xticks([]) # Hide ticks
                    ax.set_yticks([])

                # Hide unused subplots on the last page
                for j in range(num_on_page, len(axes)):
                    axes[j].set_visible(False)

                plt.suptitle(f"Grayscale Cutouts ({category}): Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                # Save the figure
                output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                plt.close(fig) # Close the figure
                print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except FileNotFoundError:
        # This catch is redundant due to the check at the start, but good to have
        print(f"!!! ERROR: File not found at {image_file_path} !!!")
    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name} (Category: {category}): {e} !!!")
        import traceback
        traceback.print_exc()

    print(f"--- Finished processing category '{category}' ---")

print("\n" + "="*80)
print(f"--- ALL cutout generation finished. Check the sub-folders inside '{base_output_dir}'. ---")
print("="*80)

--- Starting GRAYSCALE cutout generation for 6 categories (NO STRETCH) ---
Base output directory: data/star_cutouts_grayscale_pages_by_category

Processing Category: 'good'
File: data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg
Outputting to: data/star_cutouts_grayscale_pages_by_category/good
Loading grayscale JPEG image...
Image loaded: shape=(6388, 9576), dtype=uint8
Estimating background...
  Background Median: 7.00, Noise (Std Dev): 2.38
Running DAOStarFinder (FWHM=4.0, Threshold=5.0*std)...

--- Found 1937 total stars ---
Generating 20 grayscale cutout image pages (100 stars per page)...
  Saved grayscale page 1/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_001.png
  Saved grayscale page 2/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_002.png
  Saved grayscale page 3/20 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_003

In [2]:
# Import necessary libraries
import numpy as np
import cv2
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os
import math

# --- Configuration ---

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!!  UPDATE THIS PATH to the relative path of your JPG or PNG   !!!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
image_file_path = Path("data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg")
# (Using the previous 'good' image as an example)

# Define a category and output dir for this single run
category = "good_filtered_80_percent"
base_output_dir = Path("data/star_cutouts_grayscale_filtered")
output_dir = base_output_dir / category
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 5.0

# Cutout visualization parameters
cutout_size = 256       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)

# --- Main Processing Block ---
print(f"--- Starting GRAYSCALE cutout generation (Top 80% Brightness) ---")
print(f"Processing Category: '{category}'")
print(f"File: {image_file_path}")
print(f"Outputting to: {output_dir}")
print("="*80)

if not image_file_path.exists():
    print(f"!!! ERROR: File not found: {image_file_path} !!!")
    print(f"--- Please update the 'image_file_path' variable in the script. ---")
else:
    try:
        # 1. Load the JPEG image as grayscale
        print("Loading grayscale JPEG image...")
        gray_image_uint8 = cv2.imread(str(image_file_path), cv2.IMREAD_GRAYSCALE)
        if gray_image_uint8 is None:
            raise IOError(f"Could not load image file: {image_file_path}")
        print(f"Image loaded: shape={gray_image_uint8.shape}, dtype={gray_image_uint8.dtype}")
        gray_image = gray_image_uint8.astype(float) # Use float for calculations

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        if std == 0: std = np.finfo(gray_image.dtype).eps

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=threshold_sigma * std)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
        else:
            total_stars_found = len(sources)
            print(f"\n--- Found {total_stars_found} total stars (before filtering) ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # --- 5. NEW: Filter by 80% brightness ---
            if total_stars_found > 0:
                max_peak = sources['peak'][0]
                brightness_threshold = max_peak * 0.80
                
                print(f"  Brightest star peak: {max_peak:.2f}")
                print(f"  80% brightness threshold: {brightness_threshold:.2f}")
                
                # Filter the sources table
                sources = sources[sources['peak'] >= brightness_threshold]
                total_stars_filtered = len(sources)
                
                print(f"--- Keeping {total_stars_filtered} stars above 80% brightness ---")
            else:
                print("  No stars found, skipping filtering.")
            # --- End of filtering step ---

            if len(sources) == 0:
                print("\n!!! No stars remaining after 80% brightness filter. !!!")
            else:
                # 6. Paginate and create cutout images (for filtered list)
                num_pages = int(math.ceil(len(sources) / stars_per_page))
                print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

                for page_num in range(num_pages):
                    start_idx = page_num * stars_per_page
                    end_idx = min(len(sources), (page_num + 1) * stars_per_page)
                    page_sources = sources[start_idx:end_idx]
                    num_on_page = len(page_sources)

                    # Create the plot grid
                    fig, axes = plt.subplots(grid_dim, grid_dim,
                                             figsize=(1.5 * grid_dim, 1.5 * grid_dim),
                                             squeeze=False)
                    axes = axes.ravel() # Flatten for easy iteration

                    for i, source in enumerate(page_sources):
                        x, y = source['xcentroid'], source['ycentroid']
                        peak = source['peak']

                        # Define cutout boundaries
                        x_min = max(0, int(x - cutout_size // 2))
                        x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                        y_min = max(0, int(y - cutout_size // 2))
                        y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                        cutout = gray_image[y_min:y_max, x_min:x_max]

                        if cutout.size == 0:
                            axes[i].set_visible(False)
                            continue

                        # Display cutout using grayscale colormap (NO STRETCH)
                        ax = axes[i]
                        ax.imshow(cutout, origin='lower', cmap='gray', interpolation='nearest', vmin=0, vmax=255) 
                        ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                        ax.set_xticks([]) # Hide ticks
                        ax.set_yticks([])

                    # Hide unused subplots on the last page
                    for j in range(num_on_page, len(axes)):
                        axes[j].set_visible(False)

                    plt.suptitle(f"Grayscale Cutouts ({category}): Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                    # Save the figure
                    output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                    plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                    plt.close(fig) # Close the figure
                    print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name} (Category: {category}): {e} !!!")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)
print(f"--- ALL cutout generation finished. Check the folder '{output_dir}'. ---")
print("="*80)

--- Starting GRAYSCALE cutout generation (Top 80% Brightness) ---
Processing Category: 'good_filtered_80_percent'
File: data/component_output/focus/2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview.jpg
Outputting to: data/star_cutouts_grayscale_filtered/good_filtered_80_percent
Loading grayscale JPEG image...
Image loaded: shape=(6388, 9576), dtype=uint8
Estimating background...
  Background Median: 7.00, Noise (Std Dev): 2.38
Running DAOStarFinder (FWHM=4.0, Threshold=5.0*std)...

--- Found 1937 total stars (before filtering) ---
  Brightest star peak: 248.00
  80% brightness threshold: 198.40
--- Keeping 262 stars above 80% brightness ---
Generating 3 grayscale cutout image pages (100 stars per page)...
  Saved grayscale page 1/3 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_001.png
  Saved grayscale page 2/3 to 2024-10-14_03-09-54__-0.00_30.00s_0073_stretched_B_preview_grayscale_cutouts_page_002.png
  Saved grayscale page 3/3 to 2024-1

In [3]:
pip install xisf

Note: you may need to restart the kernel to use updated packages.


In [29]:
# Import necessary libraries
import numpy as np
import xisf         # Library for reading .xisf files
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from pathlib import Path
import matplotlib.pyplot as plt
import os
import math

# --- Configuration ---

# This is the local file path you provided.
image_file_path = Path("data/stretched_output/good/2025-06-11_03-55-10_North America Nebula_0.00_300.00s_0046_stretched.xisf")

# --- We will create a new output directory next to your file ---
category = "good_filtered_80_percent_xisf"
# Create the output dir in the same 'good' folder
base_output_dir = image_file_path.parent / "star_cutouts_from_xisf"
output_dir = base_output_dir / category
output_dir.mkdir(parents=True, exist_ok=True) # Create if needed

# DAOStarFinder parameters
fwhm_pixels = 4.0
threshold_sigma = 3.0 # Kept at 3.0 for better sensitivity

# Cutout visualization parameters
cutout_size = 50       # Size of the square cutout box in pixels (50x50)
stars_per_page = 100   # How many cutouts to put on each image (10x10 grid)
grid_dim = int(math.sqrt(stars_per_page)) # Grid dimensions (10x10)


def load_xisf_as_float_0_255(file_path: Path) -> np.ndarray:
    """
    Loads an XISF file and returns its image data as a 2D float
    Numpy array, rescaled to the 0.0 - 255.0 range.
    """
    print(f"Loading XISF file: {file_path}")
    
    try:
        # Instantiate the XISF object directly
        f = xisf.XISF(str(file_path))
        
        # 'read_image' returns an image object
        image_obj = f.read_image(0) 
        
        # The data is an attribute of the *returned image object*
        data = image_obj.data
        
        # --- FIX 5: Convert memoryview to a numpy array ---
        # This ensures that slicing (e.g., [:,:,0]) will work
        data = np.asarray(data)
        # --- END FIX ---
        
    except Exception as e:
        print(f"!!! Error while reading XISF file: {e} !!!")
        raise

    if data is None:
        raise ValueError("No image data found in XISF file.")
    
    # Ensure data is 2D (grayscale)
    if data.ndim == 3:
        # If it's color, convert to grayscale using standard luminance
        print("  Image is color. Converting to grayscale (luminance)...")
        # This line should now work correctly
        data = 0.299 * data[:,:,0] + 0.587 * data[:,:,1] + 0.114 * data[:,:,2]
    elif data.ndim == 2:
        print("  Image is monochrome (grayscale).")
    else:
        raise ValueError(f"Unexpected image dimensions: {data.ndim}")
        
    # Rescale data from its native range (e.g., 0.0-1.0 or 0-65535) 
    # to the 0.0 - 255.0 range that the rest of the script expects.
    print(f"  Rescaling data from [min, max] = [{data.min():.2f}, {data.max():.2f}] to [0.0, 255.0]")
    
    # Add a small epsilon to prevent division by zero if image is all black
    data_min = data.min()
    data_max = data.max()
    if (data_max - data_min) == 0:
        print("  Warning: Image is all one color (min == max). Returning as is.")
        data_norm = data - data_min # Will be all zeros
    else:
        data_norm = (data - data_min) / (data_max - data_min) 
        
    data_scaled_float = data_norm * 255.0
    
    print(f"  Final data shape: {data_scaled_float.shape}, dtype: {data_scaled_float.dtype}")
    return data_scaled_float


def process_image(image_file_path: Path):
    """
    Main function to find stars and generate cutouts.
    """
    print(f"--- Starting GRAYSCALE cutout generation (Top 80% Brightness) ---")
    print(f"Processing Category: '{category}'")
    print(f"File: {image_file_path}")
    print(f"Outputting to: {output_dir}")
    print("="*80)

    if not image_file_path.exists():
        print(f"!!! ERROR: File not found: {image_file_path} !!!")
        print(f"--- Please check that the file path is correct. ---")
        return # Exit the function
    
    try:
        # 1. Load the XISF image as a grayscale float array (0.0-255.0)
        gray_image = load_xisf_as_float_0_255(image_file_path)

        # 2. Estimate background and noise
        print("Estimating background...")
        mean, median, std = sigma_clipped_stats(gray_image, sigma=3.0, maxiters=5)
        print(f"  Background Median: {median:.2f}, Noise (Std Dev): {std:.2f}")
        
        # --- Print the threshold to help debug ---
        detection_threshold_value = threshold_sigma * std
        print(f"  Detection Threshold (sigma * std): {detection_threshold_value:.2f}")
        print(f"  (Stars will be found if their peak is > {detection_threshold_value:.2f} above the median)")

        # 3. Run DAOStarFinder
        print(f"Running DAOStarFinder (FWHM={fwhm_pixels}, Threshold={threshold_sigma}*std)...")
        data_subtracted = gray_image - median
        daofind = DAOStarFinder(fwhm=fwhm_pixels, threshold=detection_threshold_value)
        sources = daofind(data_subtracted)

        if sources is None:
            print("\n!!! No stars found with the current settings. !!!")
            print("--- Try lowering the 'threshold_sigma' value even more (e.g., 2.5) ---")
        else:
            total_stars_found = len(sources)
            print(f"\n--- Found {total_stars_found} total stars (before filtering) ---")

            # 4. Sort by brightness (descending)
            sources.sort('peak')
            sources.reverse()

            # --- 5. NEW: Filter by 80% brightness ---
            if total_stars_found > 0:
                max_peak = sources['peak'][0]
                brightness_threshold = max_peak * 0.80
                
                print(f"  Brightest star peak: {max_peak:.2f}")
                print(f"  80% brightness threshold: {brightness_threshold:.2f}")
                
                # Filter the sources table
                sources = sources[sources['peak'] >= brightness_threshold]
                total_stars_filtered = len(sources)
                
                print(f"--- Keeping {total_stars_filtered} stars above 80% brightness ---")
            else:
                print("  No stars found, skipping filtering.")
            # --- End of filtering step ---

            if len(sources) == 0:
                print("\n!!! No stars remaining after 80% brightness filter. !!!")
            else:
                # 6. Paginate and create cutout images (for filtered list)
                num_pages = int(math.ceil(len(sources) / stars_per_page))
                print(f"Generating {num_pages} grayscale cutout image pages ({stars_per_page} stars per page)...")

                for page_num in range(num_pages):
                    start_idx = page_num * stars_per_page
                    end_idx = min(len(sources), (page_num + 1) * stars_per_page)
                    page_sources = sources[start_idx:end_idx]
                    num_on_page = len(page_sources)

                    # Create the plot grid
                    fig, axes = plt.subplots(grid_dim, grid_dim,
                                             figsize=(1.5 * grid_dim, 1.5 * grid_dim),
                                             squeeze=False)
                    axes = axes.ravel() # Flatten for easy iteration

                    for i, source in enumerate(page_sources):
                        x, y = source['xcentroid'], source['ycentroid']
                        peak = source['peak']

                        # Define cutout boundaries
                        x_min = max(0, int(x - cutout_size // 2))
                        x_max = min(gray_image.shape[1], int(x + cutout_size // 2 + (cutout_size % 2)))
                        y_min = max(0, int(y - cutout_size // 2))
                        y_max = min(gray_image.shape[0], int(y + cutout_size // 2 + (cutout_size % 2)))

                        cutout = gray_image[y_min:y_max, x_min:x_max]

                        if cutout.size == 0:
                            axes[i].set_visible(False)
                            continue

                        # Display cutout using grayscale colormap (NO STRETCH)
                        # vmin=0, vmax=255 is correct because we rescaled the data
                        ax = axes[i]
                        ax.imshow(cutout, origin='lower', cmap='gray', interpolation='nearest', vmin=0, vmax=255) 
                        ax.set_title(f"#{start_idx + i + 1}\nP: {peak:.0f}", fontsize=5) # Smaller title
                        ax.set_xticks([]) # Hide ticks
                        ax.set_yticks([])

                    # Hide unused subplots on the last page
                    for j in range(num_on_page, len(axes)):
                        axes[j].set_visible(False)

                    plt.suptitle(f"Grayscale Cutouts ({category}): Page {page_num + 1}/{num_pages} (Stars {start_idx + 1}-{end_idx})\n({image_file_path.name})", fontsize=10)
                    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

                    # Save the figure
                    # Use .png for the output
                    output_filename = output_dir / f"{image_file_path.stem}_grayscale_cutouts_page_{page_num + 1:03d}.png"
                    plt.savefig(output_filename, bbox_inches='tight', dpi=150)
                    plt.close(fig) # Close the figure
                    print(f"  Saved grayscale page {page_num + 1}/{num_pages} to {output_filename.name}")

    except Exception as e:
        print(f"!!! An error occurred processing {image_file_path.name} (Category: {category}): {e} !!!")
        import traceback
        traceback.print_exc()

    print("\n" + "="*80)
    print(f"--- ALL cutout generation finished. Check the folder '{output_dir}'. ---")
    print("="*80)

# --- This makes the script runnable from the terminal ---
if __name__ == "__main__":
    process_image(image_file_path)

--- Starting GRAYSCALE cutout generation (Top 80% Brightness) ---
Processing Category: 'good_filtered_80_percent_xisf'
File: data/stretched_output/good/2025-06-11_03-55-10_North America Nebula_0.00_300.00s_0046_stretched.xisf
Outputting to: data/stretched_output/good/star_cutouts_from_xisf/good_filtered_80_percent_xisf
Loading XISF file: data/stretched_output/good/2025-06-11_03-55-10_North America Nebula_0.00_300.00s_0046_stretched.xisf
  Image is color. Converting to grayscale (luminance)...
  Rescaling data from [min, max] = [31.44, 65366.85] to [0.0, 255.0]
  Final data shape: (6388, 9576), dtype: float64
Estimating background...
  Background Median: 14.27, Noise (Std Dev): 4.38
  Detection Threshold (sigma * std): 13.13
  (Stars will be found if their peak is > 13.13 above the median)
Running DAOStarFinder (FWHM=4.0, Threshold=3.0*std)...

--- Found 8729 total stars (before filtering) ---
  Brightest star peak: 223.74
  80% brightness threshold: 178.99
--- Keeping 106 stars above 8